# Autoencoder para detecção de spam em SMS

**Disciplina:** Redes Neurais  
**Integrantes:** preencher  
**Turma:** preencher

O sistema recebe uma mensagem SMS e indica se ela parece normal (`ham`) ou suspeita de spam. O autoencoder aprende a reconstruir mensagens legítimas; erros de reconstrução acima de um limite são tratados como anomalias.

> A base escolhida contém mensagens em inglês. O protótipo não foi validado para português.


## 1. Bibliotecas e carregamento dos dados

Usaremos a SMS Spam Collection, do UCI Machine Learning Repository (5.574 mensagens rotuladas como `ham` ou `spam`).

In [ ]:
%pip -q install tensorflow scikit-learn pandas matplotlib requests

import io
import zipfile
import time
import random
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    ConfusionMatrixDisplay, classification_report
)
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Dispositivo(s):", tf.config.list_physical_devices())


## 2. Divisão dos dados

Separação estratificada: 70% treino, 15% validação e 15% teste. O vocabulário TF-IDF será ajustado somente com o treino, para evitar vazamento de dados.

In [ ]:
X = df['text']; y = df['label']
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)
print(f'Treino: {len(X_train)} ({len(X_train)/len(df):.1%})')
print(f'Validação: {len(X_val)} ({len(X_val)/len(df):.1%})')
print(f'Teste: {len(X_test)} ({len(X_test)/len(df):.1%})')
display(pd.DataFrame({'treino':y_train.value_counts(normalize=True),'validação':y_val.value_counts(normalize=True),'teste':y_test.value_counts(normalize=True)}))

## 3. Representação numérica dos textos

A regressão logística usa TF-IDF, uma representação que pondera os termos pelo quanto são informativos.

Para o autoencoder, usamos bag-of-words binário: cada posição indica se um termo ou bigrama aparece (1) ou não aparece (0) na mensagem. Essa representação é adequada para reconstruir presença/ausência de termos com uma saída sigmoid.

Os dois vocabulários são ajustados apenas no treino; validação e teste recebem somente a transformação. Não usamos embeddings contextuais nesta versão.

In [ ]:
# Baseline: vetores TF-IDF.
vectorizer = TfidfVectorizer(
    lowercase=True, strip_accents='unicode', max_features=1500,
    ngram_range=(1,2), min_df=2, max_df=0.98, norm='l2',
    dtype=np.float32
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

# Autoencoder: presença/ausência dos termos (bag-of-words binário).
bow_vectorizer = CountVectorizer(
    lowercase=True, strip_accents='unicode', binary=True,
    max_features=3000, ngram_range=(1,2), min_df=2, max_df=0.98,
    dtype=np.float32
)
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_val_bow = bow_vectorizer.transform(X_val)
X_test_bow = bow_vectorizer.transform(X_test)

print('Características TF-IDF (baseline):', X_train_tfidf.shape[1])
print('Características bag-of-words (autoencoder):', X_train_bow.shape[1])
print('Exemplo de mensagem:', X_train.iloc[0])
print('Termos presentes no vetor binário:', bow_vectorizer.get_feature_names_out()[X_train_bow[0].indices][:15])

## 4. Baseline: regressão logística

A regressão logística é uma referência tradicional e supervisionada. Ela aprende diretamente com os rótulos `ham` e `spam`; `class_weight='balanced'` ajuda a compensar o desbalanceamento.

In [ ]:
baseline = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
baseline.fit(X_train_tfidf, y_train)
pred_baseline = baseline.predict(X_test_tfidf)
score_baseline = baseline.predict_proba(X_test_tfidf)[:, list(baseline.classes_).index('spam')]
print('Baseline treinada.')

## 5. Arquitetura do autoencoder

O encoder comprime a representação binária da mensagem e o decoder tenta reconstruir quais termos estavam presentes.

**Problema da primeira versão:** com TF-IDF esparso, milhares de posições eram zero. A MSE comum podia ser dominada pelos zeros, fazendo a rede aprender uma reconstrução quase toda zerada.

**Melhoria desta versão:** o autoencoder recebe presença/ausência de termos (bag-of-words binário) e usa Binary Cross-Entropy ponderada (BCE). Termos presentes recebem peso 1; termos ausentes recebem peso 0,005. Assim, o modelo é incentivado a reconstruir os termos que realmente aparecem, em vez de otimizar principalmente os zeros.

A arquitetura é 256 → 128 → 64 → 128 → 256, com gargalo latente de 64 dimensões. A rede é treinada somente com mensagens legítimas (ham), mantendo a abordagem one-class/semi-supervisionada.

In [ ]:
# Matrizes binárias para o autoencoder.
Xtr = X_train_bow.toarray().astype(np.float32)
Xva = X_val_bow.toarray().astype(np.float32)
Xte = X_test_bow.toarray().astype(np.float32)
ytr = y_train.to_numpy()
yva = y_val.to_numpy()
yte = y_test.to_numpy()

Xtr_ham = Xtr[ytr == 'ham']
Xva_ham = Xva[yva == 'ham']
n_features = Xtr.shape[1]

# Os termos ausentes recebem peso pequeno para que os zeros não dominem a BCE.
PESO_TERMO_AUSENTE = 0.005

def bce_ponderada(y_true, y_pred):
    epsilon = tf.keras.backend.epsilon()
    y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)
    bce = -(
        y_true * tf.math.log(y_pred)
        + (1.0 - y_true) * tf.math.log(1.0 - y_pred)
    )
    pesos = tf.where(y_true > 0.0, 1.0, PESO_TERMO_AUSENTE)
    erro_por_amostra = tf.reduce_sum(pesos * bce, axis=1) / tf.reduce_sum(pesos, axis=1)
    return erro_por_amostra

inp = Input(shape=(n_features,), name='presenca_termos')
x = Dense(256, activation='relu', name='encoder_256')(inp)
x = Dense(128, activation='relu', name='encoder_128')(x)
latent = Dense(64, activation='relu', name='gargalo')(x)
x = Dense(128, activation='relu', name='decoder_128')(latent)
x = Dense(256, activation='relu', name='decoder_256')(x)
out = Dense(n_features, activation='sigmoid', name='reconstrucao')(x)
autoencoder = Model(inp, out, name='autoencoder_spam_bow')
autoencoder.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss=bce_ponderada
)
autoencoder.summary()

## 6. Treinamento do autoencoder

- Até 60 épocas, lote (batch size) de 32.
- Otimizador Adam, taxa de aprendizado 0,0005.
- BCE ponderada: peso 1 para termos presentes e 0,005 para termos ausentes.
- ReLU nas camadas ocultas e sigmoid na saída de reconstrução.
- Early Stopping com paciência 7, restaurando os melhores pesos.

A saída sigmoid reconstrói a presença de termos; ela não representa diretamente a probabilidade de spam.

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss', patience=7, restore_best_weights=True, verbose=1
)
start = time.time()
history = autoencoder.fit(
    Xtr_ham, Xtr_ham,
    validation_data=(Xva_ham, Xva_ham),
    epochs=60, batch_size=32, shuffle=True,
    callbacks=[early_stop], verbose=1
)
training_seconds = time.time() - start
print(f'Épocas executadas: {len(history.history["loss"])} | Tempo: {training_seconds:.1f} s')

plt.figure(figsize=(8,4))
plt.plot(history.history['loss'], label='Treino')
plt.plot(history.history['val_loss'], label='Validação')
plt.xlabel('Época')
plt.ylabel('BCE ponderada')
plt.title('Perda de reconstrução')
plt.legend()
plt.grid(alpha=.2)
plt.show()

def reconstruction_error(matrix):
    reconstructed = autoencoder.predict(matrix, verbose=0)
    epsilon = tf.keras.backend.epsilon()
    reconstructed = np.clip(reconstructed, epsilon, 1.0 - epsilon)
    bce = -(
        matrix * np.log(reconstructed)
        + (1.0 - matrix) * np.log(1.0 - reconstructed)
    )
    weights = np.where(matrix > 0.0, 1.0, PESO_TERMO_AUSENTE)
    return np.sum(weights * bce, axis=1) / np.sum(weights, axis=1)

val_errors_ham = reconstruction_error(Xva_ham)
threshold = float(np.percentile(val_errors_ham, 95))
val_errors_spam = reconstruction_error(Xva[yva == 'spam'])

print(f'Limite (percentil 95 ham na validação): {threshold:.8f}')
print(f'Mediana do erro ham na validação: {np.median(val_errors_ham):.8f}')

plt.figure(figsize=(8,4))
plt.hist(val_errors_ham, bins=35, alpha=.65, density=True, label='Normal (ham)')
plt.hist(val_errors_spam, bins=25, alpha=.55, density=True, label='Spam (diagnóstico)')
plt.axvline(threshold, color='red', linestyle='--', label='Limite P95 ham')
plt.xlabel('Erro de reconstrução (BCE ponderada)')
plt.ylabel('Densidade')
plt.title('Distribuição do erro na validação')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Avaliação no conjunto de teste

Erro de reconstrução acima do limite é marcado como spam. As métricas de precisão, recall e F1 são calculadas para a classe spam. O conjunto de teste só é usado nesta etapa final.

In [ ]:
errors_test = reconstruction_error(Xte)
pred_ae = (errors_test > threshold).astype(int)
y_test_bin = (yte == 'spam').astype(int)
pred_base_bin = (pred_baseline == 'spam').astype(int)
def metrics(name, y_true, pred, scores):
    return {'Modelo':name, 'Acurácia':accuracy_score(y_true,pred), 'Precisão spam':precision_score(y_true,pred,zero_division=0), 'Recall spam':recall_score(y_true,pred,zero_division=0), 'F1 spam':f1_score(y_true,pred,zero_division=0), 'ROC-AUC':roc_auc_score(y_true,scores), 'Average Precision':average_precision_score(y_true,scores)}
results = pd.DataFrame([metrics('Regressão logística',y_test_bin,pred_base_bin,score_baseline), metrics('Autoencoder',y_test_bin,pred_ae,errors_test)]).set_index('Modelo')
display(results.style.format('{:.4f}'))
print('Relatório do autoencoder:')
print(classification_report(y_test_bin,pred_ae,labels=[0,1],target_names=['Normal','Spam'],zero_division=0))
fig, ax = plt.subplots(1,2,figsize=(10,4))
ConfusionMatrixDisplay.from_predictions(y_test_bin,pred_base_bin,display_labels=['Normal','Spam'],ax=ax[0],colorbar=False); ax[0].set_title('Baseline')
ConfusionMatrixDisplay.from_predictions(y_test_bin,pred_ae,display_labels=['Normal','Spam'],ax=ax[1],colorbar=False); ax[1].set_title('Autoencoder')
plt.tight_layout(); plt.show()
print('Diferença F1 (autoencoder - baseline):', round(f1_score(y_test_bin,pred_ae)-f1_score(y_test_bin,pred_base_bin),4))

## 8. Análise de erros e demonstração

Falso positivo: mensagem legítima marcada como spam. Falso negativo: spam classificado como normal. Os exemplos são avaliados no conjunto de teste somente depois do treinamento e da definição do limite.

A célula seguinte imprime os textos completos para usar na análise do slide 13.

In [ ]:
error_table = pd.DataFrame({
    'mensagem': X_test.to_numpy(),
    'esperado': yte,
    'previsto': np.where(pred_ae == 1, 'spam', 'ham'),
    'erro': errors_test
})

print('Falsos positivos:')
display(error_table[(error_table.esperado == 'ham') & (error_table.previsto == 'spam')].head(3))
print('Falsos negativos:')
display(error_table[(error_table.esperado == 'spam') & (error_table.previsto == 'ham')].head(3))

def analisar_mensagem(texto):
    if not isinstance(texto, str) or not texto.strip():
        raise ValueError('Digite uma mensagem não vazia.')
    vetor = bow_vectorizer.transform([texto]).toarray().astype(np.float32)
    erro = float(reconstruction_error(vetor)[0])
    classe = 'SPAM (suspeita)' if erro > threshold else 'NORMAL (ham)'
    return {
        'mensagem': texto,
        'classificação': classe,
        'erro_reconstrucao': erro,
        'limite': threshold,
        'observação': 'O erro não é uma probabilidade de spam.'
    }

display(analisar_mensagem('Congratulations! You have won a free prize. Claim now!'))


In [ ]:
# Exemplos completos para o slide 13.
pd.set_option("display.max_colwidth", None)

error_table["correto"] = error_table["esperado"] == error_table["previsto"]

print("EXEMPLO DE ACERTO")
acertos = error_table[error_table["correto"]]
if not acertos.empty:
    ex = acertos.iloc[0]
    print("Mensagem:", ex["mensagem"])
    print("Esperado:", ex["esperado"])
    print("Previsto:", ex["previsto"])
else:
    print("Não houve acertos nesta execução.")

print("\nEXEMPLO DE FALSO NEGATIVO")
falsos_negativos = error_table[(error_table["esperado"] == "spam") & (error_table["previsto"] == "ham")]
if not falsos_negativos.empty:
    ex = falsos_negativos.iloc[0]
    print("Mensagem:", ex["mensagem"])
    print("Esperado:", ex["esperado"])
    print("Previsto:", ex["previsto"])
    print("Erro ponderado:", f"{ex['erro']:.8f}")
else:
    print("Não houve falsos negativos nesta execução.")

print("\nEXEMPLO DE FALSO POSITIVO")
falsos_positivos = error_table[(error_table["esperado"] == "ham") & (error_table["previsto"] == "spam")]
if not falsos_positivos.empty:
    ex = falsos_positivos.iloc[0]
    print("Mensagem:", ex["mensagem"])
    print("Esperado:", ex["esperado"])
    print("Previsto:", ex["previsto"])
else:
    print("Não houve falsos positivos nesta execução.")

## 9. Limitações, ética e referências

- A base contém SMS em inglês e é histórica; não representa todo o spam atual nem mensagens em português.
- Bag-of-words registra presença de termos, mas não compreende contexto ou intenção.
- A BCE ponderada é uma melhoria experimental; não garante que os erros de reconstrução separem ham e spam.
- O limite é uma heurística calibrada na validação. O teste permanece separado.
- Mensagens podem conter dados pessoais: minimizar coleta, proteger os dados e respeitar a legislação aplicável.

**Declaração de IA:** inteligência artificial generativa foi usada como apoio à organização, explicação dos conceitos e elaboração inicial do notebook. A equipe deve revisar, executar, adaptar e compreender o código.

**Referências:** UCI Machine Learning Repository, SMS Spam Collection (https://archive.ics.uci.edu/dataset/228/sms); Pedregosa et al., *Scikit-learn: Machine Learning in Python*, JMLR, 2011; Abadi et al., *TensorFlow: A System for Large-Scale Machine Learning*, 2016.